# TP2 — Le catalogue des livres
## Pipeline Data Complet : API Open Library → Python → Power BI
- Auteur : SELMAN Marjane
- Date : Août 2026
- Formation : B1 IA & Data — Ynov Campus Lyon
### Contexte
Data Analyst pour un groupe d'édition envisageant de racheter des librairies
indépendantes. Objectif : analyser la structure du marché du livre via l'API
Open Library pour orienter la décision du comité de direction.

### Vérification préalable de l'API
Avant l'extraction, test de l'API Open Library sur Postman pour valider sa
disponibilité et comprendre la structure de réponse JSON.
![Test Postman](test_postman.png)
Résultat : statut 200 OK confirmé, structure JSON conforme à la documentation.

In [33]:
# ==============================
# Etape 1 :  Extraction
# ==============================
# 1.1 - Exploration de l'API

In [34]:
## De 1.1.1 à 1.1.2

import requests 
import pandas as pd 
import time 

url = "https://openlibrary.org/search.json?q=fantasy&limit=5&page=1"
reponse = requests.get(url, timeout=30)
print(reponse.status_code)
data = reponse.json()
print(data)

200
{'numFound': 150861, 'start': 0, 'numFoundExact': True, 'num_found': 150861, 'documentation_url': 'https://openlibrary.org/dev/docs/api/search', 'q': 'fantasy', 'offset': None, 'docs': [{'author_key': ['OL666139A'], 'author_name': ['Carole Mortimer'], 'cover_edition_key': 'OL10738221M', 'cover_height': 3267, 'cover_i': 8337298, 'cover_width': 1833, 'ebook_access': 'printdisabled', 'edition_count': 6, 'first_publish_year': 1983, 'has_fulltext': True, 'ia': ['fantasygirl0000mort_s1b6', 'fantasygirl0000mort'], 'ia_collection': ['internetarchivebooks', 'printdisabled'], 'key': '/works/OL3775422W', 'language': ['und', 'eng'], 'public_scan_b': False, 'title': 'Fantasy Girl'}, {'author_key': ['OL661093A'], 'author_name': ['Penny Jordan'], 'cover_edition_key': 'OL10739155M', 'cover_height': 3264, 'cover_i': 6611725, 'cover_width': 1997, 'ebook_access': 'borrowable', 'edition_count': 4, 'first_publish_year': 1996, 'has_fulltext': True, 'ia': ['herchristmasfant0000jord', 'herchristmasfant00j

> *Rappel analogique et dysca* : **"data"** corresponds à l'appel complet de l'API donc toutes les données

In [35]:
## 1.1.3
print(data['docs'][0].keys())

dict_keys(['author_key', 'author_name', 'cover_edition_key', 'cover_height', 'cover_i', 'cover_width', 'ebook_access', 'edition_count', 'first_publish_year', 'has_fulltext', 'ia', 'ia_collection', 'key', 'language', 'public_scan_b', 'title'])


In [36]:
## 1.1.4
livre = data['docs'][0]
print(livre.get('title'))
print(livre.get('author_name'))
print(livre.get('author_key'))
print(livre.get('first_publish_year'))
print(livre.get('edition_count'))
print(livre.get('number_of_pages_median'))
print(livre.get('language'))
print(livre.get('ebook_access'))
print(livre.get('subject'))

Fantasy Girl
['Carole Mortimer']
['OL666139A']
1983
6
None
['und', 'eng']
printdisabled
None


> *Rappel analogique et dysca* : ".get" permet de renvoyer **None** si la clé n'existe pas sans planter 

> **Q.5** : les champs 'number_of_pages_median' et 'subject' sont absents car Open Library est une base collaborative avec des sources variée. Ainsi, le niveau de détail dépend de chaque contribution et des ressources disponibles. 

> *Ressources mobilisées durant cette partie avec le cours **Memo API Python*** : 
    1. import
    2. appel API
    3. vérification du statut de la requête
    4. voir les principales clés 

In [37]:
five_first_book = data['docs'][:5]

count_pages = 0
count_ebook = 0
for livre in five_first_book:
   if 'number_of_pages_median' in livre:
       count_pages += 1
   if 'ebook_access' in livre:
       count_ebook += 1
print(count_pages)
print(count_ebook)

0
5


> **Q1.A** :
Sur les 5 premiers livres : 0 ont `number_of_pages_median`, 5 ont `ebook_access`.
Cela montre que la complétude des données Open Library varie selon le champ. Certains sont systématiquement renseignés, d'autres non. D'où l'importance de
`.get()` pour éviter les erreurs.

In [50]:
# 1.2 - Collecte multi-sujets (8 sujets, 3 pages chacun)
sujets_livre = ['science fiction', 'roman historique', 'thriller', 'biographie', 'fantasy', 'philosophie', 'roman policier', 'développement personnel']
print(len(sujets_livre)) #juste pr verifeir que bien 8 livres

8


In [51]:
for sujet in sujets_livre:
    print(sujet)

science fiction
roman historique
thriller
biographie
fantasy
philosophie
roman policier
développement personnel


In [52]:
for sujet in sujets_livre:
    for page in range(1,4): # rappel : python s'arrete juste avant 4 pour avoir 1 a 3
        print(sujet, page)

science fiction 1
science fiction 2
science fiction 3
roman historique 1
roman historique 2
roman historique 3
thriller 1
thriller 2
thriller 3
biographie 1
biographie 2
biographie 3
fantasy 1
fantasy 2
fantasy 3
philosophie 1
philosophie 2
philosophie 3
roman policier 1
roman policier 2
roman policier 3
développement personnel 1
développement personnel 2
développement personnel 3


In [48]:
sujets_test = sujets_livre[:1]
pages_test = range(1, 2)

for sujet in sujets_test:
    for page in pages_test: 
        print(sujet, page, reponse_page.status_code) # test seul avec 1 sujet 1 apge

science fiction 1 200


In [53]:
tous_les_livres = []

url_base = "https://openlibrary.org/search.json"
for sujet in sujets_livre:
    for page in range(1,4): # rappel : python s'arrete juste avant 4 pour avoir 1 a 3
        reponse_page = requests.get(url_base, params={"q": sujet, "limit": 100, "page": page },
        timeout=30
        )
        time.sleep(1)
        
        if reponse_page.status_code == 200:
            data_page = reponse_page.json()
            for livre in data_page["docs"]:
                tous_les_livres.append({
                    "titre": livre.get("title"),
                    "premier_auteur": (livre.get("author_name") or ["Inconnu"])[0],
                    "id_auteur": (livre.get("author_key") or [None])[0],
                    "annee_publication": livre.get("first_publish_year"),
                    "nb_editions": livre.get("edition_count", 1),
                    "nb_pages_median": livre.get("number_of_pages_median", None),
                    "has_ebook": livre.get("ebook_access", "no_ebook") != "no_ebook",
                    "sujet": sujet,
                    "cle_oeuvre": livre.get("key"),
                })
        else:
            print(sujet, page, "erreur", reponse_page.status_code)

print(len(tous_les_livres))

2400


*Ressources mobilisées durant cette partie avec le cours **Memo API Python**, **internet** et **IA*** : 1. extraire liste, 2. 2eme appel API, 3. syntaxe for python, 4. pour le **try** même cours et IA. Postman pour le nom des champs 

> *Rappel analogique et dysca :*
1. **None** :
- [none] => entre crochets parce qu’il remplace une liste :si la clé manque on donne une liste de secours [none], et le [0] qui suit en tire le None
- none => pour les pages, la valeur de secours remplace un nombre, donc un None tout seul suffit
2. **parenthese** :
- elles n’ont pas le rôle d’un appel de fonction. Elles servent à regrouper : « calcule d’abord ce qu’il y a dedans, puis applique ce qui vient après »
Comme en maths : (2 + 3) × 4 donne 20, mais 2 + 3 × 4 donne 14. Les parenthèses changent l’ordre (super ex dysca)

Dans ligne : 
`(livre.get("author_name") or ["Inconnu"])[0]`
- Dans les parenthèses : « prends les auteurs, ou ["Inconnu"] s’il n’y en a pas ». ça donne une liste
- Après les parenthèses, [0] s’applique à cette liste complète
Sans elles, livre.get("author_name") or ["Inconnu"][0] appliquerait le [0] seulement à ["Inconnu"], ce qui donnerait un résultat faux

Pourquoi d’autres lignes n’en ont pas : parce qu’il n’y a rien à regrouper. Dans livre.get("title"), on ne fait qu’une seule chose.

Dans la version simplifiée, livre.get("author_name", ["Inconnu"])[0], il n’y a plus de or, donc plus besoin de parenthèses de regroupement. Les parenthèses du .get(...) sont celles de la méthode.

3. **1er élément liste en python** :
- [0]

### Gestion des erreurs de l'API (1.2)

**Constat :** lors d'un premier essai sans protection, 7 appels sur 24 ont renvoyé
une erreur 500 (erreur côté serveur Open Library) : thriller, biographie, philosophie
et roman policier étaient incomplets. Cependant, après avoir reessayer le statut était de 200 partout.(génération du code par l'IA au cas ou il serait utile plus tard...)

**Pourquoi j'ai ajouté des réessais :** un sujet incomplet fausserait mon dataset
et la réponse à la question 1.B. J'ai donc ajouté :
- une boucle de 3 essais avec une pause de 3 secondes,
- un `try / except` pour ne pas interrompre la collecte en cas de coupure de connexion,
- un message d'alerte si une page échoue malgré les 3 essais.

**Aide IA :** la structure de réessai (boucle + try/except) m'a été proposée par
Claude, avec l'explication de la différence entre une erreur 500 (le serveur répond
mal) et une coupure de connexion (il ne répond pas). Les boucles, les `params` et
l'adaptation à mon code sont de moi.

**Code secours de l'IA a implémenter au miens ultérieurement si besoin :**

```url_base = "https://openlibrary.org/search.json"

for sujet in sujets_livre:
    for page in range(1, 4):
        reussi = False
        for tentative in range(3):                       # 3 essais maximum
            try:
                reponse_page = requests.get(
                    url_base,
                    params={"q": sujet, "limit": 100, "page": page},
                    timeout=30
                )
                if reponse_page.status_code == 200:      # ça a marché : on arrête les essais
                    reussi = True
                    break
                print(sujet, page, "erreur", reponse_page.status_code, "→ nouvel essai")
            except Exception as e:                       # coupure, timeout...
                print(sujet, page, "plantage :", e)
            time.sleep(3)                                # on attend avant de réessayer

        if reussi:
            print(sujet, page, "OK")
            # l'extraction des livres viendra ici (étape D)
        else:
            print(sujet, page, "ÉCHEC après 3 essais")
        time.sleep(1)                                    # pause entre deux pages (énoncé)
```

In [ ]:
# 1.2.6

In [54]:
df_livres = pd.DataFrame(tous_les_livres)

In [56]:
df_livres.shape # test pr voir si prends bien tout le tableau

(2400, 9)

In [ ]:
# 1.2.7

In [74]:
df_livres.duplicated(subset='cle_oeuvre').sum()


np.int64(24)

In [85]:
df_livres_propre_ssdoublon = df_livres.drop_duplicates(subset='cle_oeuvre') # propre = a enlever duplicat
len(df_livres_propre_ssdoublon)
print(df_livres_propre_ssdoublon.duplicated(subset='cle_oeuvre').sum()) 
print(df_livres['cle_oeuvre'].nunique())  

0
2376


In [ ]:
# 1.2.8

In [89]:
print(len(df_livres_propre_ssdoublon) >= 500) # true car c 2376

True


Objectif de l'énoncé : au moins **500 livres uniques**.
Résultat : **2 376 livres uniques** après suppression de 24 doublons, objectif atteint

> Q1.B :
**taux de duplication moyen**
- methode 1 (pssible car j'ai déjà T des doublons et T des livres) : part/Tx100 => 24/2400x100 = 1%
- methode 2 :

`nb_sujets_par_livres = df_livres.groupby('cle_oeuvre')['sujet'].nunique()
print(len(nb_sujets_par_livres))
print(nb_sujets_par_livres.head())`

(res cellule dessous)

**sujets qui a le + et - de resultat unique**
- **biographie**, **developpement personnel**, **philosophie**, **roman historique** et **science fiction** 5 sujets qui ont le plus de resultats uniques (300) :
  

In [90]:
nb_sujets_par_livres = df_livres.groupby('cle_oeuvre')['sujet'].nunique()
print(len(nb_sujets_par_livres))
print(nb_sujets_par_livres.head())

2376
cle_oeuvre
/works/OL10004668W    1
/works/OL1000811W     1
/works/OL1000827W     1
/works/OL10014537W    1
/works/OL100261W      1
Name: sujet, dtype: int64


*Ressources mobilisées durant cette partie avec le cours **Memo API Python** et **Aides aux fonctions Python / Pandas pour le TP Clients*** : 1. .shape, 2. identfier et supprimer les doublons

In [107]:
res_unique = df_livres_propre_ssdoublon.groupby('sujet')['cle_oeuvre'].nunique()
print(res_unique.sort_values(ascending=False)) # methode qui tri valeurs ds un DF


sujet
biographie                 300
développement personnel    300
philosophie                300
roman historique           300
science fiction            300
roman policier             299
thriller                   299
fantasy                    278
Name: cle_oeuvre, dtype: int64


In [106]:
print(res_unique.sort_values(ascending=False).head(3)) # les 3 + grands
print(res_unique.sort_values(ascending=False).tail(3)) # les 3 + petits
#anticipé le cas où c'est 5000 lignes par exemple. ici seulement 8 donc facile a discerner pour ajouter dans connaissance cours

sujet
biographie                 300
développement personnel    300
philosophie                300
Name: cle_oeuvre, dtype: int64
sujet
roman policier    299
thriller          299
fantasy           278
Name: cle_oeuvre, dtype: int64
